# معمل المخرجات المهيكلة عبر `TypedDict` وفحص قدرات النماذج (`model.profile`)

أهلاً بك في المعمل العملي للمحاضرة 64!

في المعمل ده هنجرب البديل الخفيف لـ Pydantic في المخرجات المهيكلة:
1. تعريف القالب بـ **`TypedDict`** مع **`Annotated`** لوصف الحقول (واللي بيرجع قاموس بايثون عادي `dict` من غير ما يعمل تحقق صارم في وقت التشغيل).
2. تجربة التداخل في `TypedDict` مع `method="json_schema"`.
3. استكشاف خاصية **`model.profile`** العجيبة اللي بتكشف لنا مواصفات وقدرات الموديل بالكامل!


In [1]:
# استيراد الأدوات وتجهيز الموديل
from typing import List, Optional
from typing_extensions import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)
print("تم تجهيز الموديل وأدوات TypedDict بنجاح.")


تم تجهيز الموديل وأدوات TypedDict بنجاح.


### 1. استخدام `TypedDict` و `Annotated` بدل Pydantic

لاحظ هنا إننا بنستخدم `Annotated[type, ..., "description"]` لتوصيف كل حقل للموديل، والنتيجة الراجعة هتبقى `dict` عادي.


In [2]:
class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating out of 10"]

typedict_llm = model.with_structured_output(MovieDict)
res = typedict_llm.invoke("Provide details about the movie The Avengers (2012).")

print("نوع النتيجة:", type(res))
print("محتوى القاموس:", res)


نوع النتيجة: <class 'dict'>
محتوى القاموس: {'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}


### 2. القوالب المتداخلة عبر `TypedDict`

نقدر نحط قاموس `ActorDict` جوه قائمة داخل `MovieDetailsDict` (ومع `method="json_schema"` بيشتغل بسلاسة تامة على Groq!).


In [3]:
class ActorDict(TypedDict):
    name: Annotated[str, ..., "Actor name"]
    role: Annotated[str, ..., "Character name"]

class MovieDetailsDict(TypedDict):
    title: Annotated[str, ..., "Movie title"]
    year: Annotated[int, ..., "Release year"]
    cast: Annotated[List[ActorDict], ..., "Top 3 actors"]

nested_llm = model.with_structured_output(MovieDetailsDict, method="json_schema")
details = nested_llm.invoke("Details for The Avengers (2012) with top 3 actors.")

print(f"الفيلم: {details['title']} ({details['year']})")
for actor in details["cast"]:
    print(f"  - {actor['name']} بدور {actor['role']}")


الفيلم: The Avengers (2012)
  - Robert Downey Jr. بدور Tony Stark / Iron Man
  - Chris Evans بدور Steve Rogers / Captain America
  - Scarlett Johansson بدور Natasha Romanoff / Black Widow


### 3. فحص قدرات النموذج (`model.profile`)

ميزة رائعة في LangChain v1+: تقدر تفحص مواصفات النموذج الأصلي (`model.profile`) قبل ما تربطه بأدوات، لكن لاحظ إنها ما بتشتغلش لو ندهتها على الكائن الملفوف `typedict_llm.profile`!


In [4]:
print("أبرز مواصفات الموديل الحالي من model.profile:")
for k in ["name", "max_input_tokens", "max_output_tokens", "image_inputs", "reasoning_output", "tool_calling", "structured_output"]:
    print(f"  {k}: {model.profile.get(k)}")


أبرز مواصفات الموديل الحالي من model.profile:
  name: GPT OSS 20B
  max_input_tokens: 131072
  max_output_tokens: 65536
  image_inputs: False
  reasoning_output: True
  tool_calling: True
  structured_output: True


### 4. جرب بنفسك!

دلوقتي دورك للتجربة الحرة:
1. جرب تبني `TypedDict` لقائمة مهام أو ملخص رحلة سياحية (المدينة، عدد الأيام، الأنشطة المقترحة كقائمة `List[str]`).
2. لاحظ إنك تقدر تحط قيم غير مطابقة للنوع في بايثون عادي (`{"days": "five"}`) من غير ما يضرب خطأ لأن `TypedDict` لا ينفذ Validation في وقت التشغيل!


In [5]:
# جرب بنفسك هنا
class TripPlan(TypedDict):
    destination: Annotated[str, ..., "City name"]
    days: Annotated[int, ..., "Number of days"]
    highlights: Annotated[List[str], ..., "Top 3 landmarks to visit"]

planner = model.with_structured_output(TripPlan, method="json_schema")
trip = planner.invoke("Plan a 3-day trip to Luxor, Egypt with 3 highlights.")

print("الوجهة:", trip["destination"], "| المدة:", trip["days"], "أيام")
print("أهم المعالم:", trip["highlights"])


الوجهة: Luxor, Egypt | المدة: 3 أيام
أهم المعالم: ['Karnak Temple', 'Valley of the Kings', 'Luxor Temple']
